# Tier 1 Recommender Evaluation

**Note on interpretation**

This dataset is synthetic, and the catalog is local-only (Made-in-Rwanda items).
So local-presence rate should be interpreted as **local substitute coverage**, not competition against non-local SKUs.

## 1) Load catalog, queries, click_log

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import recommender as rec

DATA_DIR = Path("data")
catalog_df = pd.read_csv(DATA_DIR / "catalog.csv")
queries_df = pd.read_csv(DATA_DIR / "queries.csv")
click_log_df = pd.read_csv(DATA_DIR / "click_log.csv")

ANCHOR_COL = "synthetic_best_local_substitute_sku"
if ANCHOR_COL not in queries_df.columns:
    raise ValueError(f"Missing anchor column '{ANCHOR_COL}' in queries.csv")

print("catalog:", catalog_df.shape)
print("queries:", queries_df.shape)
print("click_log:", click_log_df.shape)

catalog: (400, 8)
queries: (120, 3)
click_log: (5000, 6)


## 2) Initialize recommender components

In [2]:
TOP_K = 5
SIMILARITY_THRESHOLD = 0.2

catalog_for_model = rec.build_text_fields(catalog_df)
rec.fit_vectorizer(catalog_for_model)
rec.POPULARITY_BY_SKU = rec._build_popularity(click_log_df)

print("Recommender initialized.")

Recommender initialized.


## 3) Run recommendations for every query

In [3]:
recs_by_query = {}
rank_rows = []

for row in queries_df.itertuples(index=False):
    qid = row.query_id
    qtext = row.query_text
    ranked = rec.recommend(
        qtext,
        top_k=TOP_K,
        similarity_threshold=SIMILARITY_THRESHOLD,
    ).reset_index(drop=True)

    recs_by_query[qid] = ranked

    for idx, item in ranked.iterrows():
        rank_rows.append(
            {
                "query_id": qid,
                "query_text": qtext,
                "rank": idx + 1,
                "sku": item.get("sku", ""),
                "score": float(item.get("score", 0.0)),
            }
        )

ranked_df = pd.DataFrame(rank_rows)
print("Queries evaluated:", len(recs_by_query))
print("Total ranked rows:", len(ranked_df))

Queries evaluated: 120
Total ranked rows: 600


## 4) Compute NDCG@5 using the synthetic best-local-substitute SKU

In [4]:
def dcg_at_k(binary_relevance, k=5):
    rel = np.array(binary_relevance[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rel.size + 2))
    return float(np.sum(rel / discounts))


def ndcg_at_5(predicted_skus, relevant_sku):
    # Single relevant anchor in synthetic data.
    binary_rel = [1 if sku == relevant_sku else 0 for sku in predicted_skus[:5]]
    return dcg_at_k(binary_rel, k=5)


ndcg_scores = []
for row in queries_df.itertuples(index=False):
    predicted = recs_by_query[row.query_id]["sku"].astype(str).tolist()
    ndcg_scores.append(ndcg_at_5(predicted, getattr(row, ANCHOR_COL)))

mean_ndcg_at_5 = float(np.mean(ndcg_scores))
print("Mean NDCG@5:", round(mean_ndcg_at_5, 4))

Mean NDCG@5: 0.3413


## 5) Compute local-presence rate for top 3

In [5]:
local_skus = set(catalog_df["sku"].astype(str))

presence_flags = []
for qid in queries_df["query_id"].astype(str):
    top3 = recs_by_query[qid]["sku"].astype(str).tolist()[:3]
    has_local = any(sku in local_skus for sku in top3)
    presence_flags.append(int(has_local))

local_presence_rate_top3 = float(np.mean(presence_flags))
print("Local-presence rate @3:", round(local_presence_rate_top3, 4))

Local-presence rate @3: 1.0


## 6) Show a small results summary table

In [6]:
avg_items_returned = float(np.mean([len(recs_by_query[qid]) for qid in recs_by_query]))

summary_df = pd.DataFrame(
    [
        {"metric": "mean_ndcg_at_5", "value": round(mean_ndcg_at_5, 4)},
        {"metric": "local_presence_rate_top3", "value": round(local_presence_rate_top3, 4)},
        {"metric": "avg_items_returned_top5_call", "value": round(avg_items_returned, 2)},
    ]
)

summary_df

,metric,value
0,mean_ndcg_at_5,0.3413
1,local_presence_rate_top3,1.0000
2,avg_items_returned_top5_call,5.0000


## 7) Show 5 example queries with returned top 5

In [7]:
examples = []
for row in queries_df.head(5).itertuples(index=False):
    top5 = recs_by_query[row.query_id]["sku"].astype(str).tolist()[:5]
    examples.append(
        {
            "query_id": row.query_id,
            "query_text": row.query_text,
            "anchor_sku": getattr(row, ANCHOR_COL),
            "top5_skus": ", ".join(top5),
        }
    )

pd.DataFrame(examples)

,query_id,query_text,anchor_sku,top5_skus
0,Q0001,leather bag for women,MIR0163,"MIR0319, MIR0146, MIR0163, MIR0384, MIR0141"
1,Q0002,cadeau en cuir pour femme,MIR0237,"MIR0243, MIR0366, MIR0253, MIR0237, MIR0248"
2,Q0003,sac cuir femme kigali,MIR0237,"MIR0243, MIR0366, MIR0253, MIR0237, MIR0248"
3,Q0004,wallet gift men,MIR0221,"MIR0221, MIR0316, MIR0077, MIR0373, MIR0250"
4,Q0005,belt leather rwanda,MIR0330,"MIR0356, MIR0171, MIR0210, MIR0385, MIR0330"
